# 🦞 YuQi 數字人生成器 - LivePortrait

這個 Notebook 可以把 YuQi 的照片變成會動的數字人影片！

**執行步驟：**
1. 依序執行所有 Cell（按 Shift+Enter）
2. 上傳 YuQi 照片
3. 選擇動作範本
4. 下載生成的影片

**執行時間：約 5-10 分鐘**

---

## ⚙️ Step 1: 檢查 GPU（必須有 GPU 才能快速運行）

In [ ]:
!nvidia-smi
print("\n如果上面顯示 GPU 資訊，代表成功！")
print("如果顯示錯誤，請前往：Runtime → Change runtime type → Hardware accelerator → GPU")

## 📦 Step 2: 安裝環境（約 2-3 分鐘）

In [ ]:
# Clone 專案
!git clone https://github.com/KwaiVGI/LivePortrait.git
%cd LivePortrait

# 安裝依賴
!pip install -r requirements.txt -q

print("✅ 環境安裝完成！")

## 📤 Step 3: 上傳 YuQi 照片

In [ ]:
from google.colab import files
import shutil
import os

print("請選擇 YuQi 的照片上傳...")
uploaded = files.upload()

# 移動到 assets 資料夾
for filename in uploaded.keys():
    shutil.copy(filename, 'assets/yuqi-photo.jpg')
    print(f"✅ 照片已上傳並儲存為: assets/yuqi-photo.jpg")
    
    # 顯示上傳的照片
    from IPython.display import Image, display
    display(Image('assets/yuqi-photo.jpg', width=300))

## 🎬 Step 4: 生成數字人影片（約 1-2 分鐘）

這裡會用內建的範例動作來驅動 YuQi 的照片

In [ ]:
# 查看所有可用的驅動影片
!ls assets/examples/driving/

print("\n可用的動作範本：")
print("- d0.mp4: 基本講話動作")
print("- d1.mp4: 點頭動作")
print("- d2.mp4: 微笑表情")
print("- d3.mp4: 其他表情")

In [ ]:
# 選擇動作範本（可以修改 d0.mp4 為其他）
DRIVING_VIDEO = "assets/examples/driving/d0.mp4"

print(f"使用動作範本: {DRIVING_VIDEO}")
print("開始生成數字人影片...")

!python inference.py \
  --source_image assets/yuqi-photo.jpg \
  --driving_video {DRIVING_VIDEO} \
  --output_dir ./yuqi_output \
  --paste_back \
  --flag_relative

print("\n✅ 影片生成完成！")

## 📺 Step 5: 預覽結果

In [ ]:
from IPython.display import Video
import os

# 找到生成的影片
output_files = [f for f in os.listdir('yuqi_output') if f.endswith('.mp4')]

if output_files:
    video_path = os.path.join('yuqi_output', output_files[0])
    print(f"找到影片: {video_path}")
    display(Video(video_path, width=400))
else:
    print("❌ 找不到生成的影片")

## 📥 Step 6: 下載結果

In [ ]:
from google.colab import files
import os
import shutil

# 列出所有生成的檔案
output_files = os.listdir('yuqi_output')
print(f"找到 {len(output_files)} 個檔案:")
for f in output_files:
    print(f"  - {f}")

# 重新命名並下載主要結果
if output_files:
    main_video = [f for f in output_files if f.endswith('.mp4')][0]
    original_path = os.path.join('yuqi_output', main_video)
    
    # 重新命名為更友善的檔名
    from datetime import datetime
    timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
    new_filename = f'yuqi-digital-human-{timestamp}.mp4'
    shutil.copy(original_path, new_filename)
    
    print(f"\n準備下載: {new_filename}")
    files.download(new_filename)
    print("✅ 下載完成！")
else:
    print("❌ 沒有找到可下載的影片")

---

## 🎨 進階功能

### 選項 A：批量生成多個動作

In [ ]:
# 生成所有內建動作
driving_videos = [
    'assets/examples/driving/d0.mp4',
    'assets/examples/driving/d1.mp4',
    'assets/examples/driving/d2.mp4',
]

for i, video in enumerate(driving_videos):
    print(f"\n處理第 {i+1}/{len(driving_videos)} 個影片...")
    output_dir = f'yuqi_batch_{i}'
    
    !python inference.py \
      --source_image assets/yuqi-photo.jpg \
      --driving_video {video} \
      --output_dir {output_dir}

print("\n✅ 批量生成完成！")

# 下載所有結果
for i in range(len(driving_videos)):
    output_dir = f'yuqi_batch_{i}'
    for f in os.listdir(output_dir):
        if f.endswith('.mp4'):
            shutil.copy(os.path.join(output_dir, f), f'yuqi-variant-{i}.mp4')
            files.download(f'yuqi-variant-{i}.mp4')

### 選項 B：上傳自己的驅動影片

In [ ]:
print("上傳你的驅動影片（MP4 格式）...")
uploaded_driving = files.upload()

for filename in uploaded_driving.keys():
    print(f"\n使用 {filename} 生成影片...")
    
    !python inference.py \
      --source_image assets/yuqi-photo.jpg \
      --driving_video {filename} \
      --output_dir ./custom_output
    
    # 下載結果
    for f in os.listdir('custom_output'):
        if f.endswith('.mp4'):
            files.download(os.path.join('custom_output', f))
    
    print("✅ 完成！")

### 選項 C：整合語音（需要 ElevenLabs API）

In [ ]:
# 安裝 moviepy 用於音訊合成
!pip install moviepy -q

from moviepy.editor import VideoFileClip, AudioFileClip

print("上傳語音檔案（MP3 格式）...")
uploaded_audio = files.upload()

for audio_file in uploaded_audio.keys():
    print(f"\n將語音合成到影片中...")
    
    # 假設已經生成了 yuqi_output 中的影片
    video_files = [f for f in os.listdir('yuqi_output') if f.endswith('.mp4')]
    if video_files:
        video_path = os.path.join('yuqi_output', video_files[0])
        
        video = VideoFileClip(video_path)
        audio = AudioFileClip(audio_file)
        
        # 調整影片長度匹配音訊
        if video.duration < audio.duration:
            video = video.loop(duration=audio.duration)
        else:
            video = video.subclip(0, audio.duration)
        
        final = video.set_audio(audio)
        final.write_videofile('yuqi-with-voice.mp4', codec='libx264', audio_codec='aac')
        
        print("✅ 語音合成完成！")
        files.download('yuqi-with-voice.mp4')
    else:
        print("❌ 找不到影片，請先執行 Step 4 生成影片")

---

## 📝 說明

**參數調整：**
- `--paste_back`: 保留原始背景（不只臉部）
- `--flag_relative`: 使用相對動作（更自然）
- `--output_size`: 調整輸出解析度（預設 512）

**如果遇到問題：**
1. 確認已開啟 GPU（Runtime → Change runtime type → GPU）
2. 照片要正面、清晰
3. 如果顯存不足，降低 output_size

**更多資訊：**
- GitHub: https://github.com/KwaiVGI/LivePortrait
- 論文: https://arxiv.org/abs/2407.03168
